# Document QnA using Gemini and Vertex AI Vector Search

## Overview

Retrieval augmented generation (RAG) has become a popular paradigm for enabling LLMs to access external data and also as a mechanism for grounding to mitigate against hallucinations.

In this notebook, you will learn how to perform RAG where you will perform Q&A over a document filled with both text and images.

### Gemini

Gemini is a family of generative AI models developed by Google DeepMind that is designed for multimodal use cases. The Gemini API gives you access to the Gemini 2.0 model.

### Comparing text-based and multimodal RAG

Multimodal RAG offers several advantages over text-based RAG:

1. **Enhanced knowledge access:** Multimodal RAG can access and process both textual and visual information, providing a richer and more comprehensive knowledge base for the LLM.
2. **Improved reasoning capabilities:** By incorporating visual cues, multimodal RAG can make better informed inferences across different types of data modalities.

This notebook shows you how to use multimodal RAG with Gemini API in Vertex AI, [text embeddings](https://cloud.google.com/vertex-ai/docs/generative-ai/model-reference/text-embeddings) to build a question answering system for a PDF document.


### Costs

This tutorial uses billable components of Google Cloud:

- Vertex AI

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

### Objectives

This notebook provides a guide to building a questions answering system using multimodal retrieval augmented generation (RAG).

You will complete the following tasks:

1. Extract data from documents containing both text and images using Gemini Vision Pro, and generate embeddings of the data, store it in vector store
2. Search the vector store with text queries to find similar text data
3. Using Text data as context, generate answer to the user query using Gemini Model.

## Getting Started

### Install Vertex AI SDK and other required packages


In [ ]:
%pip install --upgrade --quiet pymupdf langchain gradio google-cloud-aiplatform langchain_google_vertexai

In [ ]:
!pip install langchain-community

### Restart runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which restarts the current kernel.

The restart might take a minute or longer. After its restarted, continue to the next step.

In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>⚠️ Wait for the kernel to finish restarting before you continue. ⚠️</b>
</div>

### Authenticate your notebook environment (Colab only)

If you are running this notebook on Google Colab, run the cell below to authenticate your environment.

This step is not required if you are using [Vertex AI Workbench](https://cloud.google.com/vertex-ai-workbench).

In [ ]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
    # Authenticate user to Google Cloud
    from google.colab import auth

    auth.authenticate_user()

### Define Google Cloud project information and initialize Vertex AI

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
# Define project information
PROJECT_ID = "cloud-demos-gcp"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

# Initialize Vertex AI
import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION)

### Import libraries
Let's start by importing the libraries that we will need for this tutorial


In [ ]:
from datetime import datetime

# File system operations and displaying images
import os

# Import utility functions for timing and file handling
import time

# Libraries for downloading files, data manipulation, and creating a user interface
import uuid

from PIL import Image as PIL_Image
import fitz

# Initialize Vertex AI libraries for working with generative models
from google.cloud import aiplatform
import gradio as gr
import pandas as pd
from vertexai.generative_models import GenerativeModel, Image
from vertexai.language_models import TextEmbeddingModel

# Print Vertex AI SDK version
print(f"Vertex AI SDK version: {aiplatform.__version__}")

# Import LangChain components
import langchain

print(f"LangChain version: {langchain.__version__}")
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import DataFrameLoader

In [ ]:
import os
from vertexai.generative_models import GenerativeModel, Image

def extract_image_content(project_id: str, location: str, image_path: str) -> str:
    """
    Uses a Gemini Pro Vision model to extract all text and describe the contents of an image.

    Args:
        project_id (str): Your Google Cloud project ID.
        location (str): The Google Cloud region for your project (e.g., "us-central1").
        image_path (str): The local file path to the image.

    Returns:
        str: The generated text from the model, containing extracted text and a description.
    """
    # Initialize Vertex AI
    import vertexai
    vertexai.init(project=project_id, location=location)

    # --- Validate Image Path ---
    if not os.path.exists(image_path):
        return f"Error: Image file not found at '{image_path}'"

    # --- Load the Model ---
    # Use a multimodal model that supports vision
    # gemini-1.0-pro-vision-001 is a stable choice for this task.
    # gemini-1.5-pro-latest might provide even more detailed results.
    multimodal_model = GenerativeModel("gemini-2.5-flash-preview-05-20")

    # --- Prepare the Prompt ---
    # Load the image from the specified path
    image = Image.load_from_file(image_path)

    # Create the prompt. Asking for both OCR and description ensures all content is captured.
    prompt_text = """
    Analyze the image in detail. Perform the following tasks:
    1.  Extract all text visible in the image, including small print. Preserve the original formatting as much as possible.
    2.  Describe all visual elements, objects, and the overall scene.
    3.  Combine the extracted text and description into a single, comprehensive response.
    """

    # --- Generate Content ---
    try:
        # The prompt is a list containing the image and the text instruction
        response = multimodal_model.generate_content([image, prompt_text])
        return response.text
    except Exception as e:
        return f"An error occurred while calling the model: {e}"


# --- Main execution block ---
if __name__ == "__main__":
    # --- CONFIGURATION ---
    # IMPORTANT: Replace with your Google Cloud project details
    # Define project information
    PROJECT_ID = "cloud-demos-gcp"  # @param {type:"string"}
    # LOCATION = "us-central1"  # @param {type:"string"}
    # PROJECT_ID = "your-gcp-project-id"
    LOCATION = "us-central1"

    # IMPORTANT: Replace with the path to your image file
    # This can be a JPEG, PNG, etc.
    # For a good test, use an image of a receipt, a poster, or a page from a document.
    IMAGE_PATH = "fully_corrected_image.png"

    # --- RUN EXTRACTION ---
    # Check if placeholder values have been changed
    if PROJECT_ID == "your-gcp-project-id" or IMAGE_PATH == "/path/to/your/image.jpg":
        print("="*60)
        print("!!! PLEASE UPDATE PLACEHOLDER VALUES !!!")
        print(f"1. Set 'PROJECT_ID' to your Google Cloud Project ID.")
        print(f"2. Set 'IMAGE_PATH' to the actual path of your image file.")
        print("="*60)
    else:
        print(f"-> Processing image: {IMAGE_PATH}")
        extracted_content = extract_image_content(
            project_id=PROJECT_ID,
            location=LOCATION,
            image_path=IMAGE_PATH
        )

        # --- DISPLAY RESULTS ---
        print("\n" + "="*25 + " Model Output " + "="*25 + "\n")
        print(extracted_content)
        print("\n" + "="*65)


### Initializing Gemini 2.0 and Text Embedding models

In [ ]:
!pip install --upgrade google-generativeai

In [ ]:
# Loading Gemini 2.0 Model
multimodal_model = GenerativeModel("gemini-2.0-flash-001")

# Initializing embedding model
text_embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

### Download from internet a sample PDF file and default image to be shown when no results are found
[Skip this step if you have uploaded your PDF file]

---


This document describes the importance of stable power grids in Japan, highlighting the recent failure of a generator step-up transformer at the Nakoso Power Station and the rapid restoration response undertaken to maintain power supply stability.

In [ ]:
# !wget https://www.hitachi.com/rev/archive/2023/r2023_04/pdf/04a02.pdf
# !wget https://img.freepik.com/free-vector/hand-drawn-no-data-illustration_23-2150696455.jpg

# # Create an "Images" directory if it doesn't exist
# Image_Path = "./Images/"
# if not os.path.exists(Image_Path):
#     os.makedirs(Image_Path)

# !mv hand-drawn-no-data-illustration_23-2150696455.jpg {Image_Path}/blank.jpg

### Split PDF to images and extract data using Gemini Vision Pro
This module processes a set of images, extracting text and tabular data using a multimodal model (Gemini 2.0).
It handles potential errors, stores the extracted information in a DataFrame, and saves the results to a CSV file.

Create a local folder Images_pdf , PDFs

In [ ]:
import os
import time
import pandas as pd
import fitz  # PyMuPDF
from PIL import Image as PIL_Image

# Initialize Vertex AI libraries for working with generative models
import vertexai
from vertexai.generative_models import GenerativeModel, Image


def extract_image_content(model: GenerativeModel, image_path: str) -> str:
    """
    Uses a Gemini Vision model to extract all text and describe the contents of an image.

    Args:
        model (GenerativeModel): The initialized Gemini model instance.
        image_path (str): The local file path to the image.

    Returns:
        str: The generated text from the model, containing extracted text and a description.
    """
    # --- Validate Image Path ---
    if not os.path.exists(image_path):
        return f"Error: Image file not found at '{image_path}'"

    # --- Prepare the Prompt ---
    # Load the image from the specified path using the Vertex AI Image class
    image = Image.load_from_file(image_path)

    # Create the prompt. Asking for both OCR and description ensures all content is captured.
    prompt_text = """
    Analyze the image in detail. Perform the following tasks:
    1.  Extract all text visible in the image, including small print. Preserve the original formatting as much as possible.
    2.  Describe all visual elements, objects, and the overall scene.
    3.  Combine the extracted text and description into a single, comprehensive response.
    """

    # --- Generate Content ---
    try:
        # The prompt is a list containing the image and the text instruction
        response = model.generate_content([image, prompt_text])
        return response.text
    except Exception as e:
        # Return a formatted error string for easier debugging
        return f"An error occurred while calling the model: {e}"


# --- Main execution block ---
if __name__ == "__main__":
    # --- 1. CONFIGURATION ---
    # IMPORTANT: Replace with your Google Cloud project details
    PROJECT_ID = "cloud-demos-gcp"
    LOCATION = "us-central1"

    # Define directories for PDFs and the generated images
    PDF_DIRECTORY = "./PDFs/"
    IMAGES_ROOT_PATH = "./Images_pdf/"

    # --- 2. INITIALIZATION ---
    print("Initializing Vertex AI and loading the Gemini model...")
    try:
        vertexai.init(project=PROJECT_ID, location=LOCATION)
        # Load the multimodal model once to be reused for all images
        multimodal_model = GenerativeModel("gemini-2.5-flash-preview-05-20")
        print("Initialization successful.")
    except Exception as e:
        print(f"Fatal Error during Vertex AI initialization: {e}")
        print("Please ensure your GCP project ID is correct and you have authenticated.")
        exit()


    # Create the root directories if they don't exist
    os.makedirs(PDF_DIRECTORY, exist_ok=True)
    os.makedirs(IMAGES_ROOT_PATH, exist_ok=True)


    # --- 3. PDF PROCESSING LOGIC ---
    try:
        pdf_files = [f for f in os.listdir(PDF_DIRECTORY) if f.endswith(".pdf")]
        if not pdf_files:
            print(f"No PDF files found in '{PDF_DIRECTORY}'. Please add PDFs to process.")
    except FileNotFoundError:
        print(f"Error: The directory '{PDF_DIRECTORY}' was not found.")
        pdf_files = []

    # Create empty lists to store information from all PDFs
    all_page_sources = []
    all_page_contents = []
    all_page_ids = []
    global_p_id = 0

    for pdf_filename in pdf_files:
        print(f"\n--- Processing PDF: {pdf_filename} ---")

        pdf_image_path = os.path.join(IMAGES_ROOT_PATH, os.path.splitext(pdf_filename)[0])
        os.makedirs(pdf_image_path, exist_ok=True)

        # --- PDF to Image Conversion ---
        print(f"Converting PDF pages to images...")
        zoom_x, zoom_y = 2.0, 2.0
        mat = fitz.Matrix(zoom_x, zoom_y)

        try:
            doc = fitz.open(os.path.join(PDF_DIRECTORY, pdf_filename))
            # Get the number of pages before looping and closing the document
            num_pages = len(doc)
            for page in doc:
                pix = page.get_pixmap(matrix=mat)
                outpath = os.path.join(pdf_image_path, f"{os.path.splitext(pdf_filename)[0]}_{page.number}.jpg")
                pix.save(outpath)
            # Now it's safe to close the document
            doc.close()
            # Use the stored page count for the success message
            print(f"Successfully converted {num_pages} pages.")
        except Exception as e:
            print(f"Error converting PDF '{pdf_filename}' to images: {e}")
            continue

        # --- Image Processing and Content Extraction ---
        image_names = sorted(os.listdir(pdf_image_path))
        p_id, rest_count = 0, 0

        while p_id < len(image_names):
            current_image_name = image_names[p_id]
            current_image_path = os.path.join(pdf_image_path, current_image_name)

            print(f"-> Extracting content from: {current_image_name}")

            try:
                # Call the function to get content from the image
                extracted_content = extract_image_content(multimodal_model, current_image_path)

                # Check if the model returned an error message
                if extracted_content.startswith("An error occurred"):
                    raise RuntimeError(extracted_content)

                # Log progress and store results
                all_page_sources.append(current_image_path)
                all_page_contents.append(extracted_content)
                all_page_ids.append(global_p_id)

                p_id += 1
                global_p_id += 1
                rest_count = 0  # Reset retry counter on success

            except Exception as err:
                print(f"  Error processing {current_image_name}: {err}")
                print("  Taking a short break...")
                rest_count += 1
                if rest_count >= 3:
                    print(f"  Skipping image due to 3 repeated errors: {current_image_name}")
                    p_id += 1
                    rest_count = 0
                else:
                    time.sleep(5)

        print(f"--- Finished processing {pdf_filename} ---")

    # --- 4. DATAFRAME CREATION ---
    if all_page_ids:
        print("\n--- Creating Final DataFrame ---")
        df = pd.DataFrame({
            "page_id": all_page_ids,
            "page_source": all_page_sources,
            "page_content": all_page_contents
        })

        print("\n--- Final DataFrame ---")
        print(df.head())
        print(f"\nTotal pages processed: {len(df)}")

        # Optional: Save DataFrame to CSV
        # df.to_csv("extracted_pdf_content.csv", index=False)
        # print("\nDataFrame saved to extracted_pdf_content.csv")

    else:
        print("\n--- No pages were processed. ---")


In [ ]:
df.to_csv('raw_data.csv', header=True, index=False)

In [ ]:
df['page_content'][0]

# Generate Text Embeddings
Leverage a powerful language model text-embedding-005 to generate rich text embeddings that helps us find relevant information from a dataset.

In [ ]:
from langchain_community.document_loaders import DataFrameLoader
from langchain.text_splitter import CharacterTextSplitter

def generate_text_embedding(text) -> list:
    """Text embedding with a Large Language Model."""
    embeddings = text_embedding_model.get_embeddings([text])
    vector = embeddings[0].values
    return vector


# Create a DataFrameLoader to prepare data for LangChain
loader = DataFrameLoader(df, page_content_column="page_content")

# Load documents from the 'page_content' column of your DataFrame
documents = loader.load()

# Log the number of documents loaded
print(f"# of documents loaded (pre-chunking) = {len(documents)}")

# Create a text splitter to divide documents into smaller chunks
text_splitter = CharacterTextSplitter(
    chunk_size=10000,  # Target size of approximately 10000 characters per chunk
    chunk_overlap=200,  # overlap between chunks
)

# Split the loaded documents
doc_splits = text_splitter.split_documents(documents)

# Add a 'chunk' ID to each document split's metadata for tracking
for idx, split in enumerate(doc_splits):
    split.metadata["chunk"] = idx

# Log the number of documents after splitting
print(f"# of documents = {len(doc_splits)}")

texts = [doc.page_content for doc in doc_splits]
text_embeddings_list = []
id_list = []
page_source_list = []
for doc in doc_splits:
    id = uuid.uuid4()
    text_embeddings_list.append(generate_text_embedding(doc.page_content))
    id_list.append(str(id))
    page_source_list.append(doc.metadata["page_source"])
    # time.sleep(1)  # So that we don't run into Quota Issue

# Creating a dataframe of ID, embeddings, page_source and text
embedding_df = pd.DataFrame(
    {
        "id": id_list,
        "embedding": text_embeddings_list,
        "page_source": page_source_list,
        "text": texts,
    }
)
embedding_df.head()

In [ ]:
embedding_df.to_csv('data.csv',index=False, header=True)

### Creating Vertex AI: Vector Search
The code configures and deploys a vector search index on Google Cloud, making it ready to store and search through embeddings.

Embedding size :  The number of values used to represent a piece of text in vector form. Larger dimensions mean a denser and potentially more expressive representation.


Dimensions vs. Latency

* Search: Higher-dimensional embeddings can make vector similarity searches slower, especially in large databases.
* Computation: Calculations with larger vectors generally take more time during model training and inference.


In [ ]:
VECTOR_SEARCH_REGION = "us-central1"
VECTOR_SEARCH_INDEX_NAME = f"{PROJECT_ID}-vector-search-index-ht"
VECTOR_SEARCH_EMBEDDING_DIR = f"{PROJECT_ID}-vector-search-bucket-ht"
VECTOR_SEARCH_DIMENSIONS = 768

### Save the embeddings in a JSON file
To load the embeddings to Vector Search, we need to save them in JSON files with JSONL format. See more information in the docs at [Input data format and structure](https://cloud.google.com/vertex-ai/docs/matching-engine/match-eng-setup/format-structure#data-file-formats).

First, export the `id` and `embedding` columns from the DataFrame in JSONL format, and save it.

Then, create a new Cloud Storage bucket and copy the file to it.

In [ ]:
# save id and embedding as a json file
jsonl_string = embedding_df[["id", "embedding"]].to_json(orient="records", lines=True)
with open("data.json", "w") as f:
    f.write(jsonl_string)

# show the first few lines of the json file
! head -n 3 data.json

In [ ]:
# Generates a unique ID for session
UID = datetime.now().strftime("%m%d%H%M")

# Creates a GCS bucket
BUCKET_URI = f"gs://{VECTOR_SEARCH_EMBEDDING_DIR}-{UID}"
! gsutil mb -l $LOCATION -p {PROJECT_ID} {BUCKET_URI}
! gsutil cp data.json {BUCKET_URI}

### Create an Index

Now it's ready to load the embeddings to Vector Search. Its APIs are available under the [aiplatform](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform) package of the SDK.

Create an [MatchingEngineIndex](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.MatchingEngineIndex) with its `create_tree_ah_index` function (Matching Engine is the previous name of Vector Search).

In [ ]:
# create index
my_index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=f"{VECTOR_SEARCH_INDEX_NAME}",
    contents_delta_uri=BUCKET_URI,
    dimensions=768,
    approximate_neighbors_count=20,
    distance_measure_type="DOT_PRODUCT_DISTANCE",
)

By calling the `create_tree_ah_index` function, it starts building an Index. This will take under a few minutes if the dataset is small, otherwise about 50 minutes or more depending on the size of the dataset. You can check status of the index creation on [the Vector Search Console > INDEXES tab](https://console.cloud.google.com/vertex-ai/matching-engine/indexes).


#### The parameters for creating index

- `contents_delta_uri`: The URI of Cloud Storage directory where you stored the embedding JSON files
- `dimensions`: Dimension size of each embedding. In this case, it is 768 as we are using the embeddings from the Text Embeddings API.
- `approximate_neighbors_count`: how many similar items we want to retrieve in typical cases
- `distance_measure_type`: what metrics to measure distance/similarity between embeddings. In this case it's `DOT_PRODUCT_DISTANCE`

See [the document](https://cloud.google.com/vertex-ai/docs/vector-search/create-manage-index) for more details on creating Index and the parameters.


### Create Index Endpoint and deploy the Index

To use the Index, you need to create an [Index Endpoint](https://cloud.google.com/vertex-ai/docs/vector-search/deploy-index-public). It works as a server instance accepting query requests for your Index.

In [ ]:
# create IndexEndpoint
my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"{VECTOR_SEARCH_INDEX_NAME}",
    public_endpoint_enabled=True,
)

This tutorial utilizes a [Public Endpoint](https://cloud.google.com/vertex-ai/docs/vector-search/setup/setup#choose-endpoint) and does not support [Virtual Private Cloud (VPC)](https://cloud.google.com/vpc/docs/private-services-access). Unless you have a specific requirement for VPC, we recommend using a Public Endpoint. Despite the term "public" in its name, it does not imply open access to the public internet. Rather, it functions like other endpoints in Vertex AI services, which are secured by default through IAM. Without explicit IAM permissions, as we have previously established, no one can access the endpoint.

With the Index Endpoint, deploy the Index by specifying an unique deployed index ID.

In [ ]:
DEPLOYED_INDEX_NAME = VECTOR_SEARCH_INDEX_NAME.replace(
    "-", "_"
)  # Can't have - in deployment name, only alphanumeric and _ allowed
DEPLOYED_INDEX_ID = f"{DEPLOYED_INDEX_NAME}_{UID}"
# deploy the Index to the Index Endpoint
my_index_endpoint.deploy_index(index=my_index, deployed_index_id=DEPLOYED_INDEX_ID)

In [ ]:
# Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/678622647590/locations/us-central1/indexEndpoints/7966905124213751808
# Deploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/678622647590/locations/us-central1/indexEndpoints/7966905124213751808/operations/5833499451297103872
# MatchingEngineIndexEndpoint index_endpoint Deployed index. Resource name: projects/678622647590/locations/us-central1/indexEndpoints/7966905124213751808
# Creating MatchingEngineIndex
# Create MatchingEngineIndex backing LRO: projects/678622647590/locations/us-central1/indexes/9102023336543649792/operations/6066560732013527040
# MatchingEngineIndex created. Resource name: projects/678622647590/locations/us-central1/indexes/9102023336543649792
# To use this MatchingEngineIndex in another session:
# index = aiplatform.MatchingEngineIndex('projects/678622647590/locations/us-central1/indexes/9102023336543649792')


If it is the first time to deploy an Index to an Index Endpoint, it will take around 25 minutes to automatically build and initiate the backend for it. After the first deployment, it will finish in seconds. To see the status of the index deployment, open [the Vector Search Console > INDEX ENDPOINTS tab](https://console.cloud.google.com/vertex-ai/matching-engine/index-endpoints) and click the Index Endpoint.

### Ask Questions to the PDF
This code snippet establishes a question-answering (QA) system.  It leverages a vector search engine to find relevant information from a dataset and then uses the 'gemini-2.0-flash' LLM model to generate and refine the final answer to a user's query.

In [ ]:
def Test_LLM_Response(txt):
    """
    Determines whether a given text response generated by an LLM indicates a lack of information.

    Args:
        txt (str): The text response generated by the LLM.

    Returns:
        bool: True if the LLM's response suggests it was able to generate a meaningful answer,
              False if the response indicates it could not find relevant information.

    This function works by presenting a formatted classification prompt to the LLM.
    The prompt includes the original text and specific categories indicating whether sufficient information was available.
    The function analyzes the LLM's classification output to make the determination.
    """

    classification_prompt = f""" Classify the text as one of the following categories:
        -Information Present
        -Information Not Present
        Text=The provided context does not contain information.
        Category:Information Not Present
        Text=I cannot answer this question from the provided context.
        Category:Information Not Present
        Text:{txt}
        Category:"""
    classification_response = multimodal_model.generate_content(
        classification_prompt
    ).text

    if "Not Present" in classification_response:
        return False  # Indicates that the LLM couldn't provide an answer
    else:
        return True  # Suggests the LLM generated a meaningful response


def get_prompt_text(question, context):
    """
    Generates a formatted prompt string suitable for a language model, combining the provided question and context.

    Args:
        question (str): The user's original question.
        context (str): The relevant text to be used as context for the answer.

    Returns:
        str: A formatted prompt string with placeholders for the question and context, designed to guide the language model's answer generation.
    """
    prompt = """
      Answer the question using the context below. Respond with only from the text provided
      Question: {question}
      Context : {context}
      """.format(
        question=question, context=context
    )
    return prompt


def get_answer(query):
    """
    Retrieves an answer to a provided query using multimodal retrieval augmented generation (RAG).

    This function leverages a vector search system to find relevant text documents from a
    pre-indexed store of multimodal data. Then, it uses a large language model (LLM) to generate
    an answer, using the retrieved documents as context.

    Args:
        query (str): The user's original query.

    Returns:
        dict: A dictionary containing the following keys:
            * 'result' (str): The LLM-generated answer.
            * 'neighbor_index' (int): The index of the most relevant document used for generation
                                     (for fetching image path).

    Raises:
        RuntimeError: If no valid answer could be generated within the specified search attempts.
    """

    neighbor_index = 0  # Initialize index for tracking the most relevant document
    answer_found_flag = 0  # Flag to signal if an acceptable answer is found
    result = ""  # Initialize the answer string
    # Use a default image if the reference is not found
    page_source = "./Images/blank.jpg"  # Initialize the blank image
    query_embeddings = generate_text_embedding(
        query
    )  # Generate embeddings for the query

    response = my_index_endpoint.find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[query_embeddings],
        num_neighbors=5,
    )  # Retrieve up to 5 relevant documents from the vector store

    print(response)

    while answer_found_flag == 0 and neighbor_index < 4:
        context = embedding_df[
            embedding_df["id"] == response[0][neighbor_index].id
        ].text.values[
            0
        ]  # Extract text context from the relevant document

        prompt = get_prompt_text(
            query, context
        )  # Create a prompt using the question and context
        result = multimodal_model.generate_content(
            prompt
        ).text  # Generate an answer with the LLM

        if Test_LLM_Response(result):
            answer_found_flag = 1  # Exit loop when getting a valid response
        else:
            neighbor_index += (
                1  # Try the next retrieved document if the answer is unsatisfactory
            )

    if answer_found_flag == 1:
        page_source = embedding_df[
            embedding_df["id"] == response[0][neighbor_index].id
        ].page_source.values[
            0
        ]  # Extract image_path from the relevant document
    return result, page_source


query = (
    "Tell me DIE design"  # @param {type:"string"}
)

result, page_source = get_answer(query)
print(result)
print(page_source)

# Ask Questions to the PDF using Gradio UI
 this code creates a web-based frontend for your question-answering system, allowing users to easily enter queries and see the results along with relevant images.

In [ ]:
import os
import time
import pandas as pd
import fitz  # PyMuPDF
import gradio as gr
from PIL import Image as PIL_Image

# Initialize Vertex AI libraries for working with generative models
import vertexai
from google.cloud import aiplatform
from vertexai.generative_models import GenerativeModel, Image
from vertexai.language_models import TextEmbeddingModel


# --- Part 1: Core Helper Functions ---

def extract_image_content(model: GenerativeModel, image_path: str) -> str:
    """Uses a Gemini Vision model to extract content from an image."""
    if not os.path.exists(image_path):
        return f"Error: Image file not found at '{image_path}'"
    image = Image.load_from_file(image_path)
    prompt_text = """
    Analyze the image in detail. Perform the following tasks:
    1.  Extract all text visible in the image, including small print. Preserve the original formatting as much as possible.
    2.  Describe all visual elements, objects, and the overall scene.
    3.  Combine the extracted text and description into a single, comprehensive response.
    """
    try:
        response = model.generate_content([image, prompt_text])
        return response.text
    except Exception as e:
        return f"An error occurred while calling the vision model: {e}"


def generate_text_embedding(model: TextEmbeddingModel, text: str) -> list:
    """Generates a text embedding for a given string."""
    embeddings = model.get_embeddings([text])
    return embeddings[0].values


def get_rag_prompt(question: str, context: str) -> str:
    """Generates a formatted prompt for RAG."""
    return f"""
      Answer the question using the context below. Respond with only from the text provided.
      Question: {question}
      Context: {context}
      """


def is_answer_meaningful(model: GenerativeModel, response_text: str) -> bool:
    """Uses an LLM to classify if another LLM's response contained useful information."""
    classification_prompt = f"""Classify the text as one of the following categories:
        - Information Present
        - Information Not Present
        Text: The provided context does not contain information.
        Category: Information Not Present
        Text: I cannot answer this question from the provided context.
        Category: Information Not Present
        Text: {response_text}
        Category:"""
    try:
        classification_response = model.generate_content(classification_prompt).text
        return "Not Present" not in classification_response
    except Exception:
        # If classification fails, assume the answer is meaningful to be safe.
        return True


def get_answer_from_rag(
    query: str,
    index_endpoint: aiplatform.MatchingEngineIndexEndpoint,
    deployed_index_id: str,
    embedding_df: pd.DataFrame,
    language_model: GenerativeModel,
    embedding_model: TextEmbeddingModel,
) -> tuple[str, str]:
    """Retrieves an answer from the deployed index using RAG."""
    query_embeddings = generate_text_embedding(embedding_model, query)

    # Find neighbors in the deployed Matching Engine index
    response = index_endpoint.find_neighbors(
        deployed_index_id=deployed_index_id,
        queries=[query_embeddings],
        num_neighbors=5,
    )

    # Process neighbors to find a meaningful answer
    if response and response[0]:
        for neighbor in response[0]:
            # Find the original text using the neighbor's ID in our DataFrame
            context_df = embedding_df[embedding_df["id"] == neighbor.id]
            if not context_df.empty:
                context = context_df.text.values[0]
                prompt = get_rag_prompt(query, context)
                result = language_model.generate_content(prompt).text

                if is_answer_meaningful(language_model, result):
                    page_source = context_df.page_source.values[0]
                    return result, page_source  # Return the first good answer

    # Fallback response if no meaningful answer is found
    return "Could not find a relevant answer in the provided documents.", "./Images/blank.jpg"


# --- 1. CONFIGURATION ---
PROJECT_ID = "cloud-demos-gcp"
LOCATION = "us-central1"
PDF_DIRECTORY = "./PDFs/"
IMAGES_ROOT_PATH = "./Images_pdf/"

# --- Configuration for Deployed Index ---
# Replace with your Matching Engine Index Endpoint and Deployed Index ID
INDEX_ENDPOINT_RESOURCE_NAME = "projects/173195860927/locations/us-central1/indexEndpoints/3972212254736121856"
DEPLOYED_INDEX_ID = "cloud_demos_gcp_vector_search_index_ht_06161825" # Example ID, replace with your actual one

# --- 2. INITIALIZATION ---
print("Initializing Vertex AI and loading models...")
try:
    vertexai.init(project=PROJECT_ID, location=LOCATION)
    # Model for text generation (RAG)
    language_model = GenerativeModel("gemini-2.0-flash-001")
    # Model for creating text embeddings
    text_embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

    # Connect to the deployed Matching Engine endpoint
    my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint(INDEX_ENDPOINT_RESOURCE_NAME)
    print("Initialization successful.")
except Exception as e:
    print(f"Fatal Error during Vertex AI initialization: {e}")
    exit()

# --- 3. CREATE LOCAL KNOWLEDGE BASE (DATAFRAME) ---
# This DataFrame maps the document IDs to their text content.
# It's required by the RAG function to get the context for the LLM.
print("Loading local knowledge base...")
embedding_df = pd.read_csv('data.csv')
print("\n--- Local Knowledge Base DataFrame ---")
print(embedding_df.head())


# --- 4. GRADIO INTERFACE ---

def gradio_query(query_text: str) -> tuple[str, PIL_Image.Image]:
    """
    Function to be called by the Gradio interface.
    It takes a text query, gets the answer from the RAG system,
    and returns the text answer and the reference image.
    """
    # Retrieve the answer from your QA system
    result, image_path = get_answer_from_rag(
        query=query_text,
        index_endpoint=my_index_endpoint,
        deployed_index_id=DEPLOYED_INDEX_ID,
        embedding_df=embedding_df,
        language_model=language_model,
        embedding_model=text_embedding_model,
    )

    try:
        # Attempt to fetch the source image reference
        image = PIL_Image.open(image_path)  # Open the reference image
    except (FileNotFoundError, IOError):
        # Use a default image if the reference is not found or is invalid
        print(f"Warning: Image not found at '{image_path}'. Using default image.")
        image = PIL_Image.open("./Images/blank.jpg")

    return result, image  # Return both the text answer and the image


print("\nLaunching Gradio Interface...")
# Ensure a clean Gradio interface
gr.close_all()

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Document Query System with RAG and Vertex AI")
    gr.Markdown("Enter a query about your documents to get an answer and the relevant source page.")
    with gr.Row():
        with gr.Column(scale=2):
            # Input / Output Components
            query_box = gr.Textbox(label="Query", info="Enter your question here")
            process_button = gr.Button("Process", variant="primary")
            answer_box = gr.Textbox(label="Response", lines=5, interactive=False)
            clear_button = gr.Button("Clear")

        with gr.Column(scale=1):
            image_box = gr.Image(label="Reference Image", visible=True)

    # Button Click Event
    process_button.click(
        fn=gradio_query,
        inputs=query_box,
        outputs=[answer_box, image_box]
    )

    # Clear button functionality
    def clear_all():
        return "", None, ""
    clear_button.click(
        fn=clear_all,
        inputs=[],
        outputs=[query_box, image_box, answer_box]
    )


# Launch the Gradio app
demo.launch(share=True, debug=True)